In [1]:
'''
task: classify syllogism validity with TFL+ notation
models: gemma-2-2b-it, llama-3.2-3b-instruct, phi-3.5-mini-instruct
dataset: pfolio
evaluation: zero-shot
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# start preparing for QA pipeline
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 115.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


In [3]:
import pandas as pd

pfolio_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/p-folio/data/pfolio_kr_gold_train.csv")

In [4]:
# evaluation metrics

import numpy as np
import re
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def predict_answer(model, tokenizer, obj, subject, ref_relation=None, source_knowledge=None):
  # define notation grammar
  grammar = r"""
    start: program
    program: [stat]+
    stat: proposition newline
    proposition: atomicproposition | complexproposition
    complexproposition: leftparen proposition rightparen | proposition plus proposition | proposition minus proposition
    atomicproposition: term
    leftparen: "("
    rightparen: ")"
    plus: "+"
    minus: "-"
    term: plus T n | minus T n
    T: LETTER
    n: NUMBER
    newline: /\n/

    %import common.LETTER
    %import common.INT -> NUMBER
    %import common.WS
    %ignore WS
"""
  # prepare prompt
  rag_prompt = f"""
  <start_of_turn>user
  You are an expert logician. You are given a syllogism in TFL with premises between <PREMISES></PREMISES> and conclusion between <CONCLUSION></CONCLUSION> tags.
  The TFL BNF grammar to understand and reason in the language is given in the <GRAMMAR></GRAMMAR> tags.
  <GRAMMAR>{grammar}</GRAMMAR>
  <PREMISES>{subject}</PREMISES>
  <CONCLUSION>{obj}</CONCLUSION>
  Classify the conclusion as "T" if true, "F" if false or "U" if uncertain based on the premises. Present your answer only between <output></output> tags.
  <end_of_turn>
  <start_of_turn>model
  """
  input_ids = tokenizer(rag_prompt, return_tensors="pt").to(model.device)
  response = model.generate(**input_ids, max_new_tokens=500)
  predicted_relation = tokenizer.decode(response[0])
  matches = re.findall('<output>(.*)</output>', predicted_relation, flags=re.DOTALL)
  res = re.findall(r"<output>(.*)", matches[-1])  # from ['</output> tags.\n  <end_of_turn>\n  <start_of_turn>model\n  <output>T'] to ['T']
  predicted_label = res[0] if res else "None" # take first element from list ['T'] to get 'T'

  print("*** Premises: \n", subject)
  print("*** Conclusion: \n", obj)
  print("*** True Label: \n", ref_relation)
  print("*** Predicted Label: \n", predicted_label)
  return predicted_label

In [5]:
def infer_from_ontology(dataset, model, tokenizer, mode='default', notation='NL'):
  evaluation_metrics_df = pd.DataFrame(columns=["Accuracy", "Precision", "Recall", "F1"])
  reference_labels = []
  predicted_labels = []
  for index, row in dataset.iterrows():
      conclusion = row["Conclusions - " + notation]
      premises = row["Premises - " + notation]
      label = row["Truth Values"]
      if mode.lower() == "grammar":
        # conduct query with RAG retrival of sources
        # set number of candidate answers to consider as half the total triple store axioms
        source_information = """BNF GRAMMAR"""
        print("*** RAG INFORMATION:", source_information)
      # predict answer with model
      predicted_label = predict_answer(model, tokenizer, conclusion, premises, label)
      reference_labels.append(label)
      predicted_labels.append(predicted_label)
  # fill evaluation metrics dataframe
  accuracy_metric = accuracy_score(reference_labels, predicted_labels)
  precision_metric = precision_score(reference_labels, predicted_labels, average="macro")
  recall_metric = recall_score(reference_labels, predicted_labels, average="macro")
  f1_metric = f1_score(reference_labels, predicted_labels, average="macro")
  evaluation_metrics_df["Accuracy"] = [accuracy_metric]
  evaluation_metrics_df["Precision"] = [precision_metric]
  evaluation_metrics_df["Recall"] = [recall_metric]
  evaluation_metrics_df["F1"] = [f1_metric]
  print("Classification Report:", classification_report(reference_labels, predicted_labels))
  print("*************** INFERENCE COMPLETE ***************")
  return reference_labels, predicted_labels, evaluation_metrics_df, accuracy_metric, precision_metric, recall_metric, f1_metric

In [6]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

import warnings
warnings.filterwarnings("ignore")

In [7]:
# login to hugging face to have access to the model
!pip install huggingface_hub
from huggingface_hub import notebook_login
notebook_login()

In [9]:
# try rag search with gemma
tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it", device_map="auto")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [10]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='TFLPLUS')

*** Premises: 
 -(+W0-(+E0-+O0-+G0-+M0-+R0-+O0))-(+E0(+t0))-(+O0(+t0))-(+G0(+t0))-(+M0(+t0)-+R0(+t0))+W2(+t2)
*** Conclusion: 
 +O2(+t2)
*** True Label: 
 T
*** Predicted Label: 
 T
*** Premises: 
 -(+W0-(+E0-+O0-+G0-+M0-+R0-+O0))-(+E0(+t0))-(+O0(+t0))-(+G0(+t0))-(+M0(+t0)-+R0(+t0))+W2(+t2)
*** Conclusion: 
 +E2(+t2)
*** True Label: 
 F
*** Predicted Label: 
 T
*** Premises: 
 -(+W0-(+E0-+O0-+G0-+M0-+R0-+O0))-(+E0(+t0))-(+O0(+t0))-(+G0(+t0))-(+M0(+t0)-+R0(+t0))+W2(+t2)
*** Conclusion: 
 +W2(+j2)
*** True Label: 
 U
*** Predicted Label: 
 T
*** Premises: 
 +H2-(+H0-+H0)-+H2
*** Conclusion: 
 +H2-+H2
*** True Label: 
 T
*** Predicted Label: 
 T
*** Premises: 
 +C2(+b2)++I2+C2(+b2)++I2++C2(+h2)++I2++C2(+m2)++I2+(+C1(+w1)++I1++C1(+b1)++I1)+C2(+p2)+-(+I2)-((+C0++C0(+b0)++I0)--(+I0))-+((+C1+(+I1+-(+x1+b1)+-(+x1+t1)+-(+x1+t1)+-(+x1+u1))--+(-(+z1)++I1))
*** Conclusion: 
 +(+I1++I1)
*** True Label: 
 F
*** Predicted Label: 
 T
*** Premises: 
 +C2(+b2)++I2+C2(+b2)++I2++C2(+h2)++I2++C2(+m2)++I2+(

In [11]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.44518272425249167
***** PRECISION *****
0.38326135322601756
***** RECALL *****
0.39340302526332466
***** F1 *****
0.2987938129700582


,Accuracy,Precision,Recall,F1
0,0.445183,0.383261,0.393403,0.298794


In [13]:
# try rag search with llama
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", device_map="auto")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [14]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='TFLPLUS')

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 -(+W0-(+E0-+O0-+G0-+M0-+R0-+O0))-(+E0(+t0))-(+O0(+t0))-(+G0(+t0))-(+M0(+t0)-+R0(+t0))+W2(+t2)
*** Conclusion: 
 +O2(+t2)
*** True Label: 
 T
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 -(+W0-(+E0-+O0-+G0-+M0-+R0-+O0))-(+E0(+t0))-(+O0(+t0))-(+G0(+t0))-(+M0(+t0)-+R0(+t0))+W2(+t2)
*** Conclusion: 
 +E2(+t2)
*** True Label: 
 F
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

*** Premises: 
 +A2(+m2)++S2(+m2)++P2(+m2)+S2++E2++L2(+m2)+F2(+w2)++S2(+w2)+P2++D2+S2(+e2)++A2
*** Conclusion: 
 +D2
*** True Label: 
 U
*** Predicted Label: 
 T


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 +L2(+s2)++L2(+s2)++O2(+s2)++G2(+s2)++I2+N2--(+N0-+S0)
*** Conclusion: 
 +S2
*** True Label: 
 U
*** Predicted Label: 
 None


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


*** Premises: 
 +L2(+s2)++L2(+s2)++O2(+s2)++G2(+s2)++I2+N2--(+N0-+S0)
*** Conclusion: 
 +(+L1++S1)
*** True Label: 
 T
*** Predicted Label: 
 None
*** Premises: 
 +L2(+s2)++L2(+s2)++O2(+s2)++G2(+s2)++I2+N2--(+N0-+S0)
*** Conclusion: 
 -(+G0++O0--+N0)
*** True Label: 
 F
*** Predicted Label: 
 
Classification Report:                  precision    recall  f1-score   support

                      0.00      0.00      0.00         0
</output> tags.       0.00      0.00      0.00         0
              F       0.00      0.00      0.00        69
     F</output>       0.00      0.00      0.00         0
           None       0.00      0.00      0.00         0
              T       0.52      0.11      0.18       122
     T</output>       0.00      0.00      0.00         0
    T</output>        0.00      0.00      0.00         0
  T</output>");       0.00      0.00      0.00         0
   T</output>')       0.00      0.00      0.00         0
              U       0.00      0.00      0.00       1

In [15]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.04318936877076412
***** PRECISION *****
0.04727272727272727
***** RECALL *****
0.009687034277198211
***** F1 *****
0.016079158936301793


,Accuracy,Precision,Recall,F1
0,0.043189,0.047273,0.009687,0.016079


In [17]:
# try rag search with phi
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")
# CPU Enabled uncomment below 👇🏽
#model = AutoModelForCausalLM.from_pretrained("google/gemma-2b-it")
# GPU Enabled use below 👇🏽
model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct", device_map="auto")

config.json: 0.00B [00:00, ?B/s]

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [18]:
# experiment: ZS prediction without Grammar
ref_labels, pred_labels, eval_metrics_df, acc_metric, pr_metric, re_metric, f_metric = infer_from_ontology(pfolio_df, model, tokenizer, mode='default', notation='TFLPLUS')

*** Premises: 
 -(+W0-(+E0-+O0-+G0-+M0-+R0-+O0))-(+E0(+t0))-(+O0(+t0))-(+G0(+t0))-(+M0(+t0)-+R0(+t0))+W2(+t2)
*** Conclusion: 
 +O2(+t2)
*** True Label: 
 T
*** Predicted Label: 
 None
*** Premises: 
 -(+W0-(+E0-+O0-+G0-+M0-+R0-+O0))-(+E0(+t0))-(+O0(+t0))-(+G0(+t0))-(+M0(+t0)-+R0(+t0))+W2(+t2)
*** Conclusion: 
 +E2(+t2)
*** True Label: 
 F
*** Predicted Label: 
 None
*** Premises: 
 -(+W0-(+E0-+O0-+G0-+M0-+R0-+O0))-(+E0(+t0))-(+O0(+t0))-(+G0(+t0))-(+M0(+t0)-+R0(+t0))+W2(+t2)
*** Conclusion: 
 +W2(+j2)
*** True Label: 
 U
*** Predicted Label: 
 U
*** Premises: 
 +H2-(+H0-+H0)-+H2
*** Conclusion: 
 +H2-+H2
*** True Label: 
 T
*** Predicted Label: 
 U
*** Premises: 
 +C2(+b2)++I2+C2(+b2)++I2++C2(+h2)++I2++C2(+m2)++I2+(+C1(+w1)++I1++C1(+b1)++I1)+C2(+p2)+-(+I2)-((+C0++C0(+b0)++I0)--(+I0))-+((+C1+(+I1+-(+x1+b1)+-(+x1+t1)+-(+x1+t1)+-(+x1+u1))--+(-(+z1)++I1))
*** Conclusion: 
 +(+I1++I1)
*** True Label: 
 F
*** Predicted Label: 
 U
*** Premises: 
 +C2(+b2)++I2+C2(+b2)++I2++C2(+h2)++I2++C2(+m2)

In [19]:
# output results
print("***** ACCURACY *****")
print(acc_metric)
print("***** PRECISION *****")
print(pr_metric)
print("***** RECALL *****")
print(re_metric)
print("***** F1 *****")
print(f_metric)
eval_metrics_df

***** ACCURACY *****
0.13953488372093023
***** PRECISION *****
0.17116846738695482
***** RECALL *****
0.05918085240964784
***** F1 *****
0.07913924374456112


,Accuracy,Precision,Recall,F1
0,0.139535,0.171168,0.059181,0.079139


In [16]:
# empty torch cuda cache
torch.cuda.empty_cache()

# delete model from cpu
del(model)